In [1]:
import torch
import torch.nn as nn #用来搭建神经网络
import torch.optim as optim #用来更新参数
from torch.utils.data import DataLoader, TensorDataset
import mnist

In [2]:
mndata = mnist.MNIST('data/')

images, labels = mndata.load_training()
test_images, test_labels = mndata.load_testing()

x_train = torch.tensor(images, dtype=torch.float32) / 255.0
y_train = torch.tensor(labels, dtype=torch.long)#在 神经网络训练中，分类任务的标签通常需要是 torch.long 类型，因为像 nn.CrossEntropyLoss 这样的损失函数要求标签是 int64。

x_test = torch.tensor(test_images, dtype=torch.float32) / 255.0
y_test = torch.tensor(test_labels, dtype=torch.long)

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)#输入是784维度

dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(y_train[0:5])#查看前5个标签

torch.Size([60000, 784]) torch.Size([60000])
torch.Size([10000, 784]) torch.Size([10000])
tensor([9, 0, 0, 3, 0])


In [3]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128,32),
            nn.ReLU(),
            nn.Linear(32,10)
            #nn.Softmax(dim=10),交叉熵损失会自动应用 softmax，所以这里不需要显式地添加 softmax 层
        )

    def forward(self, x):
        return self.network(x)
    
model = MLP()

In [4]:
# --- 核心技能 3：检测 GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

# 将模型搬移到 GPU
model.to(device)
criterion = nn.CrossEntropyLoss()#交叉熵损失函数适用于多分类问题，能够衡量模型输出的概率分布与真实标签之间的差距。
optimizer = optim.Adam(model.parameters(), lr=0.001)


当前使用的设备: cuda


In [5]:
def test_model(model, test_loader):
    model.eval()  # 设置模型为评估模式
    torch.no_grad()  # 在评估时不计算梯度
    total, correct = 0, 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)  # 将数据搬移到 GPU
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        # 参数解析：
            #   outputs.data: 模型输出的张量。
            #   1: 指定在维度 1（即 10 个类别的方向）上取最大值。
            # 操作：寻找每一行中分数最大的那个值的索引。
            # 【返回结果维度】:
            #   _: [64] (最大值本身，在这里我们不需要，所以用下划线忽略)
            #   predicted: [64] (最大值所在的索引，即预测的类别)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

        

In [6]:
epochs = 30

for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
        test_model(model, test_loader)


        
        

Epoch [10/30], Loss: 0.3738
Test Accuracy: 88.13%
Epoch [20/30], Loss: 0.1382
Test Accuracy: 88.45%
Epoch [30/30], Loss: 0.1098
Test Accuracy: 88.56%
